# PSM Memory LoCoMo Nano Ingest

Run LoCoMo ingestion with the trained Nano PSM PyTorch checkpoint, then evaluate retrieved evidence and upload artifacts to Hugging Face.

In [ ]:
!pip install -U huggingface_hub torch
!node --version
!npm --version
import torch
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from datetime import datetime, timezone

CODE_REPO = 'https://github.com/chkrishna2001/PSM.git'
CODE_REVISION = 'main'
REPO_DIR = '/content/PSM'

HF_NANO_CHECKPOINT_REPO = 'chkrishna2001/nano-psm-primary-10m-retention-dominant-codex-selector-v3-checkpoints'
LOCAL_NANO_CHECKPOINT_DIR = '/content/nano-psm-primary-10m-retention-dominant-codex-selector-v3-checkpoints'
NANO_CHECKPOINT = f'{LOCAL_NANO_CHECKPOINT_DIR}/checkpoint-best.pt'

HF_ARTIFACT_REPO = 'chkrishna2001/psm-memory-locomo-artifacts'
RUN_ID = 'nano-v3-smoke-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
HF_RUN_DIR = f'nano/runs/{RUN_ID}'

RESULT_DIR = '/content/locomo/results'
DB = f'{RESULT_DIR}/locomo-nano-v3-smoke.db'
INGEST_LIMIT = 25
BATCH_SIZE = 5
TOP_K = 3

print('run_id', RUN_ID)
print('hf_artifacts', f'hf://{HF_ARTIFACT_REPO}/{HF_RUN_DIR}')

## Download Code And Checkpoint

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

!rm -rf {REPO_DIR}
!git clone {CODE_REPO} {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin
!git checkout {CODE_REVISION}
!git pull --ff-only || true
!git rev-parse --short HEAD

checkpoint_dir = snapshot_download(
    repo_id=HF_NANO_CHECKPOINT_REPO,
    repo_type='model',
    local_dir=LOCAL_NANO_CHECKPOINT_DIR,
    force_download=True,
)
print('checkpoint_dir', checkpoint_dir)
assert Path(NANO_CHECKPOINT).exists(), NANO_CHECKPOINT
assert Path('benchmark/locomo/src/ingest-nano.ts').exists(), 'Repo does not include ingest-nano yet. Push local changes first.'
assert Path('nano-psm/src/nano_psm/predict.py').exists(), 'Repo does not include nano predictor yet. Push local changes first.'

## Build And Preflight

In [ ]:
from pathlib import Path
import json

!npm install
!npm run build
!python -m compileall nano-psm/src/nano_psm

assert Path('benchmark/locomo/data/locomo10.json').exists()
assert Path('dist/benchmark/locomo/src/ingest-nano.js').exists()
assert Path('dist/benchmark/locomo/src/evaluate.js').exists()
print('preflight ok')

## Run Nano Ingestion Smoke

In [ ]:
!mkdir -p {RESULT_DIR}
!rm -f {DB} {DB}-wal {DB}-shm

!node dist/benchmark/locomo/src/ingest-nano.js \
  --data benchmark/locomo/data/locomo10.json \
  --db {DB} \
  --nano-checkpoint {NANO_CHECKPOINT} \
  --nano-config nano-psm/configs/primary-10m.json \
  --limit {INGEST_LIMIT} \
  --batch-size {BATCH_SIZE} \
  --device auto

!cp benchmark/locomo/results/ingest-nano-summary.json {RESULT_DIR}/ingest-nano-summary.json


## Evaluate Evidence Retrieval

In [ ]:
EVAL_OUT = f'{RESULT_DIR}/locomo-nano-v3-smoke-results.json'

!node dist/benchmark/locomo/src/evaluate.js \
  --data benchmark/locomo/data/locomo10.json \
  --db {DB} \
  --out {EVAL_OUT} \
  --top-k {TOP_K}

summary = json.loads(Path(EVAL_OUT).read_text(encoding='utf-8'))['summary']
print(json.dumps(summary, indent=2))

## Upload Artifacts To Hugging Face

In [ ]:
from huggingface_hub import create_repo, upload_folder
from pathlib import Path
import shutil

artifact_root = Path('/content/locomo/hf-artifacts') / HF_RUN_DIR
artifact_root.mkdir(parents=True, exist_ok=True)
for path in Path(RESULT_DIR).glob('*'):
    if path.is_file():
        shutil.copy2(path, artifact_root / path.name)

create_repo(repo_id=HF_ARTIFACT_REPO, repo_type='dataset', private=True, exist_ok=True)
upload_folder(
    repo_id=HF_ARTIFACT_REPO,
    repo_type='dataset',
    folder_path=str(artifact_root),
    path_in_repo=HF_RUN_DIR,
    commit_message=f'upload nano locomo smoke {RUN_ID}',
)
print('uploaded', f'hf://{HF_ARTIFACT_REPO}/{HF_RUN_DIR}')
!find {artifact_root} -maxdepth 1 -type f -printf '%f %k KB\n' | sort

## Optional Full Run

After the smoke ingest and retrieval outputs look sane, set `INGEST_LIMIT = 0`, change `DB` and `RUN_ID`, rerun ingestion/evaluation, then run answer evaluation from `benchmark/locomo/psm-memory-locomo-colab.ipynb` against the full Nano-ingested DB.